# 3.1 - Brand classification. 
This notebook follows after applying the `../model appliers/2 -apply_angletag.ipynb` notebook.

In a third phase the notebooks will focus on predicting the correct 'brand' of a car. These brands are tags that were present at the scraping phase and no inference or propagation is required. 

There are multiple notebooks in this phase - each referenced to by a second digit in the format of `x.y - Brand classification` where `x` coincides with the phase and will always be three and `y` coincides with a unique notebook being used within this phase. References in this phase will always have this `x.y` notation. Within a notebook sections are labeled with `x.y.z. <title>`; the `x` and `y` values are constant in the notebook; the `z` element is the section within the notebook. 

Three different approaches will be explored in phase 3; each approach has it's own distinct model being trained. Given the long time it took to train 20.000 images in phase 2 (up to 45 minutes) for a relatively small set - this phase will explore techniques that come to results more quickly than making a model based on the KERAS API from scratch. 

 - notebook `3.2. - Brand classification: transfer learning (angled).ipynb` will use the angle tags generated in phase two and output one model per angle. The full prediction pipeline is then: 
    - Apply YOLO box
    - assess Usability - open for discussion
    - predict angle
    - use model of trained angle to predict brand. 

- notebook `3.3. - Brand classification: transfer learning (unangled).ipynb` will not use the angle tags and will be used to train the full dataset; the goal is to predict one brand per image. The full prediction pipeline up until this point is then: 
    - Apply YOLO box
    - assess Usability - open for discussion
    - use model of trained angle to predict brand. 

- notebook `3.4. - Brand classification: reinforcement learning.ipynb` will use reinforcement learning to try and predict the image - this is not a standard approach to image recognition problems, but it was an experiment I wanted to try based on the interviews being made in newspapers following the DeepSeek launch: 
    > Nieuwe trainingsmethode van DeepSeek “Een andere fundamentele verandering die DeepSeek heeft geïntroduceerd is de manier waarop AI-modellen worden getraind. Waar de meeste Large Language Models vertrouwen op enorme hoeveelheden gelabelde data en Supervised Fine-Tuning, heeft DeepSeek-R1 laten zien dat dit sterke redeneervermogen ook kan worden bereikt met een aanpak die puur gebaseerd is op Reinforcement Learning (RL).
    
    Source: [manners.nl](https://www.manners.nl/deepseek-ai-muur-budget-big-tech-open-source/)

    > Why it matters: Reinforcement learning has surprising utility in training large language models to reason. As researchers press models into service in more complex tasks — math, coding, animated graphics, and beyond — reinforcement learning is emerging as an important path to progress.

    Source: [deeplearning.ai](https://www.deeplearning.ai/the-batch/how-deepseek-r1-and-kimi-k1-5-use-reinforcement-learning-to-improve-reasoning/)


- notebook `3.5. - Brand scoring.ipynb` will use the output of notebooks `3.2`, `3.3` and `3.4` to asses the best approach and pitch the performance of an establish method (transfer learning) against a method that's not typically recommended for image classification tasks: ((reinforcement learning)). In the final scoring notbeook we'll also aim to answer the question whether or not it's worth to train an angle classifier and use models trained on less datapoints, but more homogenous points versus a model that was trained on a large batch of data that's more heterogenous.


The difficulty will be to use the same data and do this efficiently in three different notebooks. (i.e. do a single pass of augmentation versus 3 separate augmentation phases). Have the same splits being applied everywhere etc... 

To guarantee this; the current notebook will handle all augmentation and splitting needs for the model. The output of this notebook is a CSV dump of all files with augmented data being generated and included in notebook `3.1`. 

Since we plan on using `resnet50` in notebooks `3.2` and `3.3` that means we have to use a fixed training shape of `224 * 224`, for the sake of uniformity this shape requirement is used across ALL notebooks in phase three - including the `3.4` notebook where RL is used.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import sys
#import re
import random
import uuid
from tqdm import tqdm


sys.path.append('../../utils')
import config_handling as conf
import cnn_helpers
import file_io

2025-03-08 09:30:26.171651: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
basedir, db  = conf.applyconf('../../config/automotive.conf.ini')

Connection established


In [5]:
augment_base = os.path.join(basedir, 'augmentated data', 'brand phase')
augment_csv_dump = os.path.join(basedir, 'CSV-data', 'brand phase')


## 3.1.1. Wipe slate: 
Clean out the augmentation folder and wipe the augmented data.

In [ ]:
file_io.wipe_folder(augment_base)
file_io.wipe_folder(augment_csv_dump)

## 3.1.2 Making distribution decisions
we need some way of making our classes balanced, we know from the EDA phase that there are heavy imbalances between certain brands of the cars in this dataset. For instance BMW is a very popuplar brand whereas Lotus or Alpine are lesser known brands and thus less prevalent in the dataset.

We'll adress this by using augmentation again, when training the angle model it becam apparant that the model performance could shift about 0.7% depending on the random choices made at various points in the training process (which images to train/test split, which images to augmentate, what values to use in the augmentation process). To prevent this kind of drift between the notebooks `3.2`, `3.3` and `3.4`; we'll use the current notebook `3.1` to do all splitting operations, and all augmentation operations. 

This has the added benefit of being more efficient: The augmentation happens ONCE and is then re-used across different notebooks.

In [ ]:
#what we learned from phase 1: 
BINMODELS_PASS = 2
BINMODELS_MINSCORE = 0.9

#how many samples: per brand per angle
BRAND_ANGLE_COMBO_SAMPLES = 10000##let me think about this for a sec